# 05 — National Robustness Check

## Why this notebook exists

Berlin was selected as the project city for data volume, cycling share, OSM
coverage, and the existence of a Vision Zero policy framework — not because it
was expected to differ from other cities.

The truck finding rests on 767 Berlin crashes and 32 fatalities. Small counts
invite a fair question: is this a real mechanism, or an artefact of one city's
sample? This notebook answers it against all German bicycle crashes, where
fatality counts are two orders of magnitude larger.

## What is being tested

Three questions, in order:

1. **Does the severity mechanism replicate?** If truck involvement carries the
   same elevated KSI rate nationally as in Berlin, the Berlin estimate is not
   noise.
2. **Does the share of deaths transfer?** Berlin's headline — trucks account for
   39% of cyclist deaths — is the number most likely to be misquoted as a national
   fact.
3. **If it does not transfer, why?** A difference in the denominator would mean
   the mechanism is intact but the context differs.

## Coverage constraint

National row counts rise 77% between 2016 (151,673) and 2019 (268,370) and then
plateau at 256,000–273,000. That rise reflects federal states being progressively
added to the dataset, not a rise in accidents. `IstGkfz` is also absent from the
2017 release.

The comparison window is therefore **2019–2025**, where coverage is complete and
the truck flag is present in every file. This differs from the Berlin analysis
window (2018–2025), so figures quoted from the two are not directly
interchangeable — a difference noted wherever both appear.

## Read-only

This notebook loads the raw releases and does not write anything. It touches
neither `src/` nor the processed dataset.

In [1]:
import glob
import math
import os

import numpy as np
import pandas as pd


def wilson_ci(successes, n, z=1.96):
    """Wilson score interval. Behaves sensibly for small counts, unlike the
    normal approximation."""
    if n == 0:
        return (float("nan"), float("nan"))
    p = successes / n
    denom = 1 + z**2 / n
    centre = (p + z**2 / (2 * n)) / denom
    margin = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return (centre - margin, centre + margin)


def two_proportion_ztest(successes, totals):
    x1, x2 = successes
    n1, n2 = totals
    p_pool = (x1 + x2) / (n1 + n2)
    se = math.sqrt(p_pool * (1 - p_pool) * (1 / n1 + 1 / n2))
    z = (x1 / n1 - x2 / n2) / se
    return z, math.erfc(abs(z) / math.sqrt(2))


# Column names drift across the ten annual releases; harmonised against the
# official Unfallatlas Datensatzbeschreibung.
COLUMN_MAP = {
    "IstStrasse": "STRZUSTAND",
    "IstStrassenzustand": "STRZUSTAND",
    "LICHT": "ULICHTVERH",
    "IstSonstig": "IstSonstige",
    "UIDENTSTLAE": "UIDENTSTLA",
}

paths = sorted(
    p for p in glob.glob("data/Unfallorte*.txt") + glob.glob("data/Unfallorte*.csv")
    if "LinRef" not in p
)
if not paths:
    raise FileNotFoundError(
        "No raw Unfallatlas files in data/. Download the 2016-2025 accident files "
        "from https://unfallatlas.statistikportal.de (skip the LinRef ones)."
    )

frames = []
for p in paths:
    d = pd.read_csv(p, sep=";", encoding="utf-8-sig", low_memory=False)
    d = d.rename(columns=COLUMN_MAP)
    frames.append(d)
    print(f"  {os.path.basename(p):25s} {len(d):>8,}")

df = pd.concat(frames, ignore_index=True)
df["UJAHR"] = pd.to_numeric(df["UJAHR"], errors="raise").astype("int16")
print(f"\n{len(df):,} rows total")

  Unfallorte.txt             151,673
  Unfallorte2017.txt         195,229
  Unfallorte2018.txt         211,868
  Unfallorte2019.txt         268,370
  Unfallorte2020.csv         237,994
  Unfallorte2022.csv         256,492
  Unfallorte2023.csv         269,048
  Unfallorte2024.csv         268,519
  Unfallorte_2021.txt        238,826
  Unfallorte_2025.csv        273,007

2,371,026 rows total


---
## 1. Does the severity mechanism replicate?

The comparison is like-for-like: truck crashes against car-only crashes, both
bicycle–motor vehicle collisions, so it is not distorted by single-bicycle falls.

Berlin figures for reference: truck KSI 27.25% [24.22–30.51], car-only 10.59%
[10.22–10.97], risk ratio 2.57×. Truck turning crashes are 1.1% of crashes and 31%
of fatalities.

In [2]:
# Coverage constraint: national row counts rise 77% from 2016 to 2019 as federal
# states were progressively added, and IstGkfz is absent from the 2017 release.
# The comparison window is therefore 2019-2025, where coverage is complete.
WINDOW = (2019, 2025)

nat = df[
    (df["IstRad"] == 1)
    & df["UJAHR"].between(*WINDOW)
    & df["IstGkfz"].notna()
].copy()

nat["is_ksi"] = nat["UKATEGORIE"].isin([1, 2]).astype(int)
nat["is_fatal"] = (nat["UKATEGORIE"] == 1).astype(int)

print(f"National bicycle crashes {WINDOW[0]}-{WINDOW[1]}: {len(nat):,}")
print(f"Fatalities: {nat['is_fatal'].sum():,}  |  KSI: {nat['is_ksi'].sum():,}")
print(f"KSI base rate:              {nat['is_ksi'].mean():.4f}")
print(f"Overall fatality rate:      {nat['is_fatal'].mean():.4%}")

print("\nRows per year:")
print(nat["UJAHR"].value_counts().sort_index().to_string())

National bicycle crashes 2019-2025: 575,348
Fatalities: 2,820  |  KSI: 97,481
KSI base rate:              0.1694
Overall fatality rate:      0.4901%

Rows per year:
UJAHR
2019    74549
2020    78187
2021    77316
2022    82052
2023    87180
2024    86543
2025    89521


In [3]:
# Like-for-like: both groups are bicycle-motor vehicle collisions, so the
# comparison is not distorted by single-bicycle falls.
truck = nat[nat["IstGkfz"] == 1]
car = nat[(nat["IstPKW"] == 1) & (nat["IstGkfz"] == 0)]

rows = []
for label, frame in [("Truck", truck), ("Car only", car)]:
    n = len(frame)
    ksi, fat = int(frame["is_ksi"].sum()), int(frame["is_fatal"].sum())
    k_lo, k_hi = wilson_ci(ksi, n)
    f_lo, f_hi = wilson_ci(fat, n)
    rows.append({
        "group": label, "crashes": n,
        "ksi_rate": ksi / n, "ksi_lo": k_lo, "ksi_hi": k_hi,
        "fatal": fat, "fatal_rate": fat / n, "fat_lo": f_lo, "fat_hi": f_hi,
    })

summary = pd.DataFrame(rows)
print(summary.to_string(index=False, float_format=lambda v: f"{v:.5f}"))

rr_ksi = summary.loc[0, "ksi_rate"] / summary.loc[1, "ksi_rate"]
rr_fat = summary.loc[0, "fatal_rate"] / summary.loc[1, "fatal_rate"]
print(f"\nKSI risk ratio (truck vs car-only):  {rr_ksi:.2f}x")
print(f"Fatality risk ratio:                 {rr_fat:.2f}x")

# The headline asymmetry
share_crashes = len(truck) / len(nat)
fat_total, fat_truck = int(nat["is_fatal"].sum()), int(truck["is_fatal"].sum())
print(f"\nTrucks are {share_crashes:.1%} of national bicycle crashes")
print(f"but account for {fat_truck / fat_total:.1%} of cyclist fatalities "
      f"({fat_truck:,} of {fat_total:,})")

# Mechanism
TURNING = 2  # UTYP1 = 2, Abbiege-Unfall
tt = truck[truck["UTYP1"] == TURNING]
print(f"\nTruck turning crashes: {len(tt):,} ({len(tt) / len(nat):.2%} of all)")
print(f"  fatalities: {int(tt['is_fatal'].sum()):,} "
      f"({int(tt['is_fatal'].sum()) / fat_total:.1%} of the national total)")
print(f"  fatality rate within this group: {tt['is_fatal'].mean():.2%}")
print(f"\nTurning share of truck crashes:    {(truck['UTYP1'] == TURNING).mean():.1%}")
print(f"Turning share of car-only crashes: {(car['UTYP1'] == TURNING).mean():.1%}")

   group  crashes  ksi_rate  ksi_lo  ksi_hi  fatal  fatal_rate  fat_lo  fat_hi
   Truck     6481   0.28776 0.27687 0.29891    328     0.05061 0.04553 0.05622
Car only   305575   0.12640 0.12522 0.12758   1176     0.00385 0.00364 0.00407

KSI risk ratio (truck vs car-only):  2.28x
Fatality risk ratio:                 13.15x

Trucks are 1.1% of national bicycle crashes
but account for 11.6% of cyclist fatalities (328 of 2,820)

Truck turning crashes: 2,214 (0.38% of all)
  fatalities: 147 (5.2% of the national total)
  fatality rate within this group: 6.64%

Turning share of truck crashes:    34.2%
Turning share of car-only crashes: 24.6%


### 1.1 The mechanism replicates

| | Berlin (2018–2025) | Germany (2019–2025) |
|---|---|---|
| Truck crashes | 767 | 6,481 |
| Truck KSI rate | 27.25% [24.22–30.51] | **28.78% [27.69–29.89]** |
| Car-only KSI rate | 10.59% [10.22–10.97] | 12.64% [12.52–12.76] |
| KSI risk ratio | 2.57× | 2.28× |
| Truck turning fatality rate | 5.75% | 6.64% |

The confidence intervals for the truck KSI rate overlap across a sample 8.4 times
larger. The Berlin estimate is not a small-sample artefact.

### 1.2 But the share of deaths does not transfer

| | Berlin | Germany |
|---|---|---|
| Truck share of crashes | 2.0% | **1.1%** |
| Truck share of cyclist deaths | 39.1% | **11.6%** |
| Overall cyclist fatality rate | 0.195% | **0.490%** |
| Turning share of truck crashes | 52.7% | 34.2% |

Trucks account for 39.1% of cyclist deaths in Berlin but 11.6% nationally — a
3.4-fold difference in a figure that describes the same mechanism.

**The denominator explains it.** Nationally, 0.490% of bicycle crashes kill the
cyclist; in Berlin, 0.195%. National figures include rural roads at 70–100 km/h,
where speed supplies many other fatal mechanisms. Two of Berlin's own features also
differ: cyclists encounter trucks in crashes almost twice as often (2.0% vs 1.1%),
and Berlin's truck crashes are far more turning-dominated (52.7% vs 34.2%), which
is what a dense junction network produces.

So Berlin's 39.1% is not an anomaly to be explained away — it is what the same
mechanism looks like when comparatively little else is killing cyclists.

**The claim to make is "39% of cyclist deaths in Berlin", never "in Germany".**

---
## 2. Is this an urban pattern or a Berlin peculiarity?

If the denominator argument holds, other dense cities should show the same
elevated truck share of deaths. Germany's three city-states have their own `ULAND`
codes and can be isolated directly.

Individual cities cannot be ranked — fatality counts run from 20 to 64 and
confidence intervals span 30–40 percentage points. Pooling is what gives the
statistical power that city-level comparison cannot.

In [4]:
CITY_STATES = {11: "Berlin", 2: "Hamburg", 4: "Bremen"}

print("City-states individually:\n")
for code, name in CITY_STATES.items():
    g = nat[nat["ULAND"] == code]
    tk = g[g["IstGkfz"] == 1]
    fat, tk_fat = int(g["is_fatal"].sum()), int(tk["is_fatal"].sum())
    lo, hi = wilson_ci(tk_fat, fat)
    print(f"{name:8s} crashes={len(g):>7,}  deaths={fat:>4}  "
          f"overall fatality={g['is_fatal'].mean():>6.3%}  "
          f"truck share of deaths={tk_fat/fat:>6.1%} [{lo:.1%}-{hi:.1%}]  "
          f"turning share={(tk['UTYP1']==2).mean():>6.1%}  "
          f"truck share of crashes={tk['IstGkfz'].count()/len(g):>6.2%}")

# Pool the city-states against everything else. Individual cities have too few
# fatalities to place in an order; pooling gives the power that ranking cannot.
is_cs = nat["ULAND"].isin(CITY_STATES)

print("\nPooled:\n")
for label, g in [("City-states", nat[is_cs]), ("Rest of Germany", nat[~is_cs])]:
    tk = g[g["IstGkfz"] == 1]
    fat, tk_fat = int(g["is_fatal"].sum()), int(tk["is_fatal"].sum())
    lo, hi = wilson_ci(tk_fat, fat)
    print(f"{label}")
    print(f"  crashes={len(g):>7,}  deaths={fat:>5,}")
    print(f"  overall fatality rate  = {g['is_fatal'].mean():.3%}")
    print(f"  truck share of deaths  = {tk_fat/fat:.1%} [{lo:.1%}-{hi:.1%}]")
    print(f"  turning share of truck crashes = {(tk['UTYP1']==2).mean():.1%}\n")

City-states individually:

Berlin   crashes= 32,756  deaths=  64  overall fatality=0.195%  truck share of deaths= 39.1% [28.1%-51.3%]  turning share= 52.7%  truck share of crashes= 1.98%
Hamburg  crashes= 18,965  deaths=  45  overall fatality=0.237%  truck share of deaths= 31.1% [19.5%-45.7%]  turning share= 44.7%  truck share of crashes= 1.14%
Bremen   crashes=  7,255  deaths=  20  overall fatality=0.276%  truck share of deaths= 20.0% [8.1%-41.6%]  turning share= 51.9%  truck share of crashes= 1.46%

Pooled:

City-states
  crashes= 58,976  deaths=  129
  overall fatality rate  = 0.219%
  truck share of deaths  = 33.3% [25.8%-41.8%]
  turning share of truck crashes = 50.8%

Rest of Germany
  crashes=516,372  deaths=2,691
  overall fatality rate  = 0.521%
  truck share of deaths  = 10.6% [9.5%-11.8%]
  turning share of truck crashes = 31.2%



### 2.1 An urban pattern, not a Berlin anomaly

| | City-states | Rest of Germany |
|---|---|---|
| Crashes | 58,976 | 516,372 |
| Cyclist deaths | 129 | 2,691 |
| Overall fatality rate | **0.219%** | **0.521%** |
| Truck share of deaths | **33.3% [25.8–41.8]** | **10.6% [9.5–11.8]** |
| Turning share of truck crashes | 50.8% | 31.2% |

The intervals do not overlap — 25.8% against 11.8% leaves a 14-point gap. Two
things move together: cyclists are roughly half as likely to die in a crash in a
city-state, and trucks account for 3.1× as large a share of the deaths that
remain.

This is the denominator argument confirmed. In a dense low-speed city
comparatively little else kills cyclists, so trucks dominate what is left. Turning
conflicts are correspondingly more dominant (50.8% vs 31.2%), which is what a
dense junction network produces.

Individually the three cannot be ranked:

| | Crashes | Deaths | Truck share of deaths | Truck share of crashes |
|---|---|---|---|---|
| Berlin | 32,756 | 64 | 39.1% [28.1–51.3] | **1.98%** |
| Hamburg | 18,965 | 45 | 31.1% [19.5–45.7] | 1.14% |
| Bremen | 7,255 | 20 | 20.0% [8.1–41.6] | 1.46% |

All three intervals overlap heavily, and Bremen's rests on 20 deaths spanning
33 percentage points. No ordering between them is claimed or supported.

**One metric does support comparison**, because it rests on thousands of crashes
rather than tens of deaths: cyclists are involved in crashes with trucks in 1.98%
of Berlin cases against 1.14% in Hamburg — 1.7× as often, despite Hamburg hosting
one of Europe's largest ports. Whether this reflects Berlin's cycling share,
construction activity, or heavy-vehicle route management cannot be separated
without exposure data, but it is a measurable difference and a concrete question
for the city.

---
## 3. Conclusions

1. **The severity mechanism replicates.** Truck KSI rate 28.78% [27.69–29.89]
   nationally against 27.25% [24.22–30.51] in Berlin, on a sample 8.4× larger.
   Overlapping intervals. The Berlin finding is not a small-sample artefact.

2. **The share of deaths is an urban pattern.** 33.3% in the city-states against
   10.6% elsewhere, non-overlapping intervals, with half the overall fatality
   rate. Berlin's 39.1% sits within the expected spread.

3. **The claim to make is "39% of cyclist deaths in Berlin", never "in Germany".**

4. **Berlin's truck encounter rate is genuinely higher** — 1.98% of crashes
   against 1.14% in Hamburg. This is the one figure that supports a comparison
   between cities, and it raises a question rather than answering one.

**Windows.** The national comparison uses 2019–2025, where national coverage is
complete and `IstGkfz` is present in every release. The Berlin analysis elsewhere
in this project uses 2018–2025, the years Berlin appears in the dataset. Berlin's
truck share of deaths is 43.2% over 2018–2025 and 39.1% over 2019–2025; the
matched-window figure is used in every comparison here.

**Limitations.** Every comparison is conditional on a crash having occurred, so
these measure severity rather than collision probability — no cycling exposure data
is used, and none is implied. Fatality counts in the city-states are small (129
pooled, 20 in Bremen). Trucks operate disproportionately on arterial roads with
higher speed limits, so part of the severity gap reflects road environment rather
than vehicle type.